# 📋 Pipeline de Transcription et Résumé Syndical

Ce notebook traite un fichier audio de 10h (réunion syndicat/direction) avec :
- Transcription avec WhisperX (GPU) — modèle `large-v3`
- Diarisation des locuteurs via Pyannote 3.1
- Génération d'un Procès-Verbal administratif avec `qwen3.5:9b` via Ollama

## ⚠️ Prérequis
Installer les dépendances **une seule fois** dans ton terminal avant de lancer ce notebook :
```bash
pip install torch==2.8.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
pip install git+https://github.com/m-bain/whisperx.git ollama pydub av
```

## ⚠️ Gestion de la VRAM (16 Go — RTX 2000 Ada)
WhisperX et Ollama ne tournent **pas** simultanément.
Un nettoyage strict de la VRAM sépare les deux étapes.

## 🐛 Corrections appliquées
1. **Config définie AVANT les imports** → évite le `NameError: MODEL_OLLAMA`
2. **Patch PyAV re-appliqué dans la cellule 2** → évite que WhisperX repasse par ffmpeg.exe bloqué
3. **Guards de sécurité** sur transcription vide et chunks vides
4. **Debug prints** pour diagnostiquer facilement les problèmes futurs

---
## ⚙️ CELLULE 1 : Variables globales et patch PyAV

In [ ]:
import os, gc
import torch
import numpy as np
import av

# =============================================================
# --- CONFIGURATION (DOIT ÊTRE DÉFINIE EN PREMIER) ---
# =============================================================
AUDIO_DIR          = "sources"          # Dossier contenant tous les fichiers audio
SUPPORTED_FORMATS  = (".m4a", ".mp3", ".wav", ".flac", ".ogg", ".mp4")
HF_TOKEN           = os.getenv('HF_TOKEN')
DEVICE             = "cuda"
MODEL_WHISPER      = "large-v3"
LANGUAGE           = "fr"               # Langue audio (évite auto-détection)
COMPUTE_TYPE       = "int8_float16"
WHISPER_BATCH_SIZE = 8
GAP_THRESHOLD      = 0.5               # Seuil pour combler les 'trous' de diarisation
MODEL_OLLAMA       = "qwen3.5:9b"
OUTPUT_DIR         = "resultats"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(AUDIO_DIR, exist_ok=True)

# =============================================================
# --- PATCH PyAV : remplace ffmpeg.exe (contourne AppLocker) ---
# IMPORTANT : appliqué ici ET dans la cellule 2 (les imports
# whisperx dans la cellule 2 écrasent le patch si non réappliqué)
# =============================================================
def load_audio_direct(file_path: str, sr: int = 16000):
    container = av.open(file_path)
    stream = container.streams.audio[0]
    resampler = av.AudioResampler(format='s16', layout='mono', rate=sr)
    frames = []
    for frame in container.decode(stream):
        for r_frame in resampler.resample(frame):
            frames.append(r_frame.to_ndarray())
    for r_frame in resampler.resample(None):
        frames.append(r_frame.to_ndarray())
    container.close()
    if not frames:
        return np.zeros((0,), dtype=np.float32)
    return np.concatenate(frames, axis=1).flatten().astype(np.float32) / 32768.0

# Application du patch APRÈS import de whisperx
import whisperx
import whisperx.audio
whisperx.audio.load_audio = load_audio_direct
whisperx.load_audio = load_audio_direct
print("✅ Patch PyAV actif — ffmpeg.exe contourné")

# =============================================================
# --- Détection des fichiers audio ---
# =============================================================
audio_files = sorted([
    os.path.join(AUDIO_DIR, f)
    for f in os.listdir(AUDIO_DIR)
    if f.lower().endswith(SUPPORTED_FORMATS)
])

if not audio_files:
    raise FileNotFoundError(f"❌ Aucun fichier audio trouvé dans '{AUDIO_DIR}'. Formats supportés : {SUPPORTED_FORMATS}")

if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU : {torch.cuda.get_device_name(0)} ({vram:.1f} Go VRAM)")
else:
    raise SystemError("❌ CUDA non détecté. Vérifiez vos pilotes NVIDIA.")

print(f"✅ Configuration prête — Modèle Whisper : {MODEL_WHISPER} ({LANGUAGE}) | Ollama : {MODEL_OLLAMA}")
print(f"📁 {len(audio_files)} fichier(s) audio détecté(s) :")
for f in audio_files:
    print(f"   - {f}")

---
## 🎙️ CELLULE 2 : Transcription et Diarisation (étape GPU intensive)

In [ ]:
import torch, gc, os
import whisperx
import av
import numpy as np
import pandas as pd
from whisperx.diarize import DiarizationPipeline

# =============================================================
# CORRECTIF CLÉ : Re-appliquer le patch PyAV ici
# Les imports ci-dessus rechargent whisperx et écrasent
# le patch appliqué en cellule 1 → audio vide sans ce fix
# =============================================================
import whisperx.audio
whisperx.audio.load_audio = load_audio_direct
whisperx.load_audio = load_audio_direct
print("✅ Patch PyAV re-appliqué dans cellule 2")

# =============================================================
# --- Fonctions utilitaires ---
# =============================================================
def format_timestamp(seconds: float) -> str:
    td = int(seconds)
    h, m, s = td // 3600, (td % 3600) // 60, td % 60
    return f"{h:02d}:{m:02d}:{s:02d}"

def format_srt_timestamp(seconds: float) -> str:
    ms = int((seconds - int(seconds)) * 1000)
    td = int(seconds)
    h, m, s = td // 3600, (td % 3600) // 60, td % 60
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"

def save_srt(segments, srt_path):
    with open(srt_path, "w", encoding="utf-8") as f:
        for i, seg in enumerate(segments, 1):
            start = format_srt_timestamp(seg.get('start', 0))
            end = format_srt_timestamp(seg.get('end', 0))
            speaker = seg.get('speaker', 'INCONNU')
            text = seg.get('text', '').strip()
            f.write(f"{i}\n{start} --> {end}\n[{speaker}] {text}\n\n")

def fill_diarization_gaps(diar_segments, total_duration, gap_threshold=0.5):
    gap_segments = []
    if not diar_segments:
        return [{'start': 0.0, 'end': total_duration, 'speaker': 'INCONNU'}]
    if diar_segments[0]['start'] > gap_threshold:
        gap_segments.append({'start': 0.0, 'end': diar_segments[0]['start'], 'speaker': 'INCONNU'})
    for i in range(len(diar_segments) - 1):
        gs, ge = diar_segments[i]['end'], diar_segments[i+1]['start']
        if ge - gs > gap_threshold:
            gap_segments.append({'start': gs, 'end': ge, 'speaker': 'INCONNU'})
    if total_duration - diar_segments[-1]['end'] > gap_threshold:
        gap_segments.append({'start': diar_segments[-1]['end'], 'end': total_duration, 'speaker': 'INCONNU'})
    return gap_segments

# =============================================================
# --- Pipeline principal ---
# =============================================================
print(f"🚀 Démarrage pipeline sur {DEVICE}...")
print(f"📥 Chargement Whisper {MODEL_WHISPER} et Diarisation...")
model = whisperx.load_model(MODEL_WHISPER, DEVICE, compute_type=COMPUTE_TYPE)
diarize_model = DiarizationPipeline(model_name="pyannote/speaker-diarization-3.1", token=HF_TOKEN, device=DEVICE)

for file_idx, audio_file in enumerate(audio_files, 1):
    file_name = os.path.basename(audio_file)
    file_label = os.path.splitext(file_name)[0]
    print(f"\n🎤 [{file_idx}/{len(audio_files)}] Traitement de {file_name}...")

    # Chargement audio
    audio = whisperx.load_audio(audio_file)
    print(f"   📊 Audio chargé : {len(audio)/16000:.1f} secondes ({len(audio)} samples)")

    if len(audio) == 0:
        print(f"   ⚠️ ATTENTION : audio vide pour {file_name} — fichier ignoré.")
        continue

    # Transcription
    result = model.transcribe(audio, batch_size=WHISPER_BATCH_SIZE, language=LANGUAGE)
    print(f"   📊 Segments bruts WhisperX : {len(result.get('segments', []))}")

    if not result.get("segments"):
        print(f"   ⚠️ ATTENTION : transcription vide pour {file_name} — fichier ignoré.")
        continue

    # Alignement
    model_a, metadata = whisperx.load_align_model(language_code=result["language"], device=DEVICE)
    result = whisperx.align(result["segments"], model_a, metadata, audio, DEVICE, return_char_alignments=False)
    del model_a; gc.collect(); torch.cuda.empty_cache()
    print(f"   📊 Segments après alignement : {len(result.get('segments', []))}")

    # Diarisation
    diarize_output = diarize_model(audio)
    diar_segments = []
    for _, row in diarize_output.iterrows():
        diar_segments.append({'start': row['start'], 'end': row['end'], 'speaker': row['speaker']})
    print(f"   📊 Locuteurs détectés : {len(set(s['speaker'] for s in diar_segments))}")

    total_duration = len(audio) / 16000
    gap_segments = fill_diarization_gaps(diar_segments, total_duration, GAP_THRESHOLD)
    if gap_segments:
        diar_segments.extend(gap_segments)
        diar_segments.sort(key=lambda x: x['start'])

    diarize_df = pd.DataFrame(diar_segments)
    result = whisperx.assign_word_speakers(diarize_df, result)

    # Sauvegarde TXT
    txt_lines = []
    for seg in result["segments"]:
        spk = seg.get("speaker", "INCONNU")
        ts = f"[{format_timestamp(seg['start'])}]"
        txt_lines.append(f"{ts} {spk}: {seg['text']}\n")

    txt_path = f"{OUTPUT_DIR}/{file_label}_transcription.txt"
    with open(txt_path, "w", encoding="utf-8") as f:
        f.writelines(txt_lines)
    print(f"   📄 TXT sauvegardé : {txt_path} ({len(txt_lines)} lignes)")

    # Sauvegarde SRT
    srt_path = f"{OUTPUT_DIR}/{file_label}.srt"
    save_srt(result["segments"], srt_path)
    print(f"   ✅ Terminée. TXT : {txt_path} | SRT : {srt_path}")

del model, diarize_model; gc.collect(); torch.cuda.empty_cache()
print(f"\n🧹 VRAM libérée. Tous les fichiers ({len(audio_files)}) ont été traités.")

**Structure du dossier :**
```
projet/
├── sources/
│   ├── reunion_matin.m4a
│   ├── reunion_aprem.m4a
│   └── ...
├── resultats/
└── pipeline_transcription.ipynb
```

---
## 🧹 CELLULE 3 : Nettoyage STRICT de la VRAM

In [ ]:
import gc, torch

print("🧹 Nettoyage VRAM pour libérer la place à Ollama...")

for var_name in ['model', 'model_a', 'diarize_model']:
    if var_name in globals():
        del globals()[var_name]

gc.collect()
torch.cuda.empty_cache()

print(f"✅ VRAM après nettoyage : {torch.cuda.memory_reserved()/1e9:.2f} Go réservés")
print("✅ Prêt pour Ollama.")

---
## 📝 CELLULE 4 : Génération du PV avec Ollama (qwen3.5:9b)

In [ ]:
import ollama, os, time

OPTIONS_SEGMENT = {"temperature": 0.1, "num_predict": 6000, "num_ctx": 8192}
OPTIONS_RESUME  = {"temperature": 0.2, "num_predict": 6000, "num_ctx": 16384}
LINES_PER_CHUNK = 80

def get_ollama_content(resp):
    if hasattr(resp, 'message'): return resp.message.content
    if isinstance(resp, dict) and 'message' in resp: return resp['message']['content']
    return str(resp)

print(f"🚀 Génération des PV avec {MODEL_OLLAMA}...")

for audio_file in audio_files:
    file_label = os.path.splitext(os.path.basename(audio_file))[0]
    txt_path = f"{OUTPUT_DIR}/{file_label}_transcription.txt"

    if not os.path.exists(txt_path):
        print(f"⚠️ Fichier transcription introuvable : {txt_path} — skipping.")
        continue

    print(f"\n📂 Traitement de : {file_label}")
    with open(txt_path, "r", encoding="utf-8") as f:
        lines = f.readlines()

    total_chars = sum(len(l) for l in lines)
    print(f"   📄 Transcription : {len(lines)} lignes, {total_chars} caractères")

    if not lines or total_chars < 10:
        print(f"   ⚠️ Fichier transcription vide ou quasi-vide ! "
              f"Vérifiez que la cellule 2 a bien produit du contenu. Skipping.")
        continue

    chunks = ["".join(lines[i:i+LINES_PER_CHUNK]) for i in range(0, len(lines), LINES_PER_CHUNK)]
    print(f"📦 {len(chunks)} segments à traiter...")

    notes = []
    for i, chunk in enumerate(chunks, 1):
        clean_chunk = chunk.strip()

        if not clean_chunk:
            print(f"📝 Segment {i}/{len(chunks)}... ⚠️ chunk vide, ignoré.")
            continue

        print(f"📝 Segment {i}/{len(chunks)} ({len(clean_chunk)} car.)... ", end="", flush=True)
        t_seg = time.time()
        try:
            prompt = (
                "/nothink\n"
                "Tu es un greffier chargé de retranscrire mot pour mot la substance d'une réunion syndicale.\n\n"
                "CONSIGNES STRICTES :\n"
                "1. Discours rapporté au présent (ex: \"Le président rappelle que\", \"UFAP répond que\").\n"
                "2. Chaque intervention ou paragraphe DOIT commencer explicitement par le nom de l'organisation "
                "ou de l'orateur suivi du verbe (ex: \"[PRÉSIDENT] indique que\", \"[CGT] souligne que\").\n"
                "3. Identifie correctement qui parle grâce aux balises de locuteur (ex: SPEAKER_00). "
                "Si possible, déduis les rôles d'après le contexte si ce n'est pas explicite.\n"
                "4. Note TOUS les arguments, même mineurs. Ne supprime rien.\n"
                "5. Note précisément les chiffres, dates, noms de lieux, références réglementaires.\n"
                "6. Retranscris CHAQUE prise de parole, même courte. Tu peux synthétiser les banalités "
                "mais ne fusionne pas les arguments d'intervenants différents.\n"
                "7. Ne fais aucune introduction globale (\"Voici le résumé...\"). Démarre immédiatement.\n"
                "8. Longueur attendue : au moins 300 mots par segment.\n\n"
                f"Segment ({i}/{len(chunks)}) :\n{clean_chunk}\n\nRetranscription détaillée :"
            )
            resp = ollama.chat(
                model=MODEL_OLLAMA,
                messages=[{"role": "user", "content": prompt}],
                options=OPTIONS_SEGMENT,
                think=False
            )
            note = get_ollama_content(resp)
            notes.append(note)
            print(f"✅ ({time.time()-t_seg:.0f}s, {len(note)} car. générés)")
        except Exception as e:
            print(f"⚠️ Erreur : {e}")

    if not notes:
        print(f"   ⚠️ Aucune note générée pour {file_label}. PV non créé.")
        continue

    # Sauvegarde PV détaillé
    pv_path = f"{OUTPUT_DIR}/PV_DETAIL_{file_label}.txt"
    with open(pv_path, "w", encoding="utf-8") as f:
        f.write(f"PV DÉTAILLÉ — {file_label}\n{'='*60}\n\n")
        for i, note in enumerate(notes, 1):
            f.write(f"\n--- SEGMENT {i}/{len(notes)} ---\n\n{note}\n")
    print(f"   📄 PV sauvegardé : {pv_path}")

    # Résumé global
    print("📋 Résumé global... ", end="", flush=True)
    all_notes_text = "\n\n".join(notes)
    if len(all_notes_text) > 30000:
        all_notes_text = all_notes_text[:30000]
    try:
        prompt_res = (
            "/nothink\n"
            "Extrais les POINTS CLÉS de ces notes de réunion syndicale.\n\n"
            "CONSIGNES :\n"
            "- Liste chaque décision, engagement, désaccord et question restée ouverte.\n"
            "- Format : liste à puces, une puce par point en identifiant CLAIREMENT qui a dit quoi "
            "(ex: \"Le directeur a annoncé que...\", \"L'UFAP a réclamé...\").\n"
            "- Conserve les noms d'organisations et les chiffres exacts.\n"
            "- Maximum 15 points clés pour ce groupe.\n\n"
            f"Notes de réunion :\n{all_notes_text}\n\nPoints clés :"
        )
        resp_res = ollama.chat(
            model=MODEL_OLLAMA,
            messages=[{"role": "user", "content": prompt_res}],
            options=OPTIONS_RESUME,
            think=False
        )
        resume_content = get_ollama_content(resp_res)
        resume_path = f"{OUTPUT_DIR}/RESUME_{file_label}.txt"
        with open(resume_path, "w", encoding="utf-8") as f:
            f.write(f"RÉSUMÉ — {file_label}\n{'='*60}\n\n" + resume_content)
        print(f"✅ ({len(resume_content)} car.)")
        print(f"   📋 Résumé sauvegardé : {resume_path}")
    except Exception as e:
        print(f"⚠️ Erreur résumé : {e}")

print("\n✅ PV et Résumés générés avec succès !")

---
## 🏁 CELLULE 5 : Nettoyage final

In [ ]:
import gc, torch, os

print("🧹 Nettoyage final...")

try:
    torch.cuda.empty_cache()
    gc.collect()
    print(f"✅ VRAM finale : {torch.cuda.memory_reserved()/1e9:.2f} Go réservés")
except Exception as e:
    print(f"✅ Nettoyage terminé ({e})")

print("\n✅ Pipeline terminé avec succès !")
print("📁 Fichiers générés :")
for f in sorted(os.listdir(OUTPUT_DIR)):
    filepath = os.path.join(OUTPUT_DIR, f)
    size_kb = os.path.getsize(filepath) / 1024
    tag = ""
    if f.startswith("PV_DETAIL_"):
        tag = " 📄 (PV exhaustif — fait foi)"
    elif f.startswith("RESUME_"):
        tag = " 📋 (résumé rapide)"
    elif f.endswith("_transcription.txt"):
        tag = " 🎤 (transcription brute)"
    print(f"   - {filepath} ({size_kb:.1f} Ko){tag}")